# MIE312 Lab 1 — Open Channel Flow

Developed by Prof. Sullivan (pierre.sullivan@utoronto.ca) If you wish to suggest improvements (and please do make suggestions), please send them to the email address above.


This notebook is a  report-friendly version of the lab workbook for **Lab 1: Open Channel Flow**.

It is designed so you can:
- enter your raw measurements once,
- compute all key quantities automatically,
- compare theory and experiment,
- generate polished plots for your report,
- produce a short discussion-ready summary at the end.

### How to use in Colab
1. Open this notebook from the link in the lab manual and choose **File → Save a copy in Drive**.
2. Run the setup cell.
3. Replace each `np.nan` in the measurement tables with your readings.
4. Run the analysis cells under each section.
5. Run Section E to download the figures and tables for your report.


## Lab context

The lab handout states that the goals are to:
- verify Bernoulli's equation,
- study the effect of a submerged sill on open-channel flow,
- generate and recognize a hydraulic jump.

The flume width used in this notebook is set to **53 mm**.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", None)

# Constants
g = 9.81              # m/s^2
rho = 1000.0          # kg/m^3
flume_width = 53e-3   # m

plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = True

print("Setup complete.")
print(f"Flume width = {flume_width:.3f} m")

# Folder for the figures and tables saved for your report (emptied each time this cell runs)
import shutil
from pathlib import Path
OUT_DIR = Path("lab1_outputs")
shutil.rmtree(OUT_DIR, ignore_errors=True)
OUT_DIR.mkdir()

def _has_finite_data(fig):
    """True if any line, marker or bar in the figure has a finite value."""
    for ax in fig.axes:
        for line in ax.get_lines():
            if np.isfinite(np.asarray(line.get_ydata(), float)).any():
                return True
        for coll in ax.collections:
            offsets = np.asarray(coll.get_offsets(), float)
            if offsets.size and np.isfinite(offsets).all(axis=1).any():
                return True
        for patch in ax.patches:
            if np.isfinite(patch.get_height()) and patch.get_height() != 0:
                return True
    return False

def save_fig(name):
    """Save the current figure as a 300-dpi PNG for your report (called just before plt.show())."""
    fig = plt.gcf()
    if not _has_finite_data(fig):
        return  # nothing plotted yet, so don't save an empty figure
    if globals().get("USE_EXAMPLE_DATA", False):
        fig.text(0.5, 0.5, "EXAMPLE DATA", fontsize=48, color="red", alpha=0.2,
                 ha="center", va="center", rotation=30, transform=fig.transFigure)
    fig.savefig(OUT_DIR / f"{name}.png", dpi=300, bbox_inches="tight")

## Helper functions

In [ ]:

def L_min_to_m3_s(Q_L_min):
    return Q_L_min / 1000 / 60


def mm_to_m(x_mm):
    return x_mm / 1000


def velocity_rectangular_channel(Q_m3_s, width_m, depth_m):
    return Q_m3_s / (width_m * depth_m)


def froude_number(V, y):
    return V / np.sqrt(g * y)


def safe_percent_error(measured, theoretical):
    return 100 * (measured - theoretical) / theoretical


## Example data (optional)
The measurement tables in this notebook start blank: replace each `np.nan` with your own reading.

To see how the notebook works before you have data, set `USE_EXAMPLE_DATA = True` below and choose **Runtime → Run all**. The example values then replace your tables, every saved figure is stamped **EXAMPLE DATA**, and the summary is flagged. **Set it back to `False` before you analyze your own measurements.**

In [ ]:
USE_EXAMPLE_DATA = False

# Example measurements (used only when USE_EXAMPLE_DATA = True)
EXAMPLE_DATA = {
    "bernoulli": pd.DataFrame({"trial": [1, 2], "H_mm": [55.0, 92.0],
                               "h_mm": [10.0, 8.0], "Q_L_min": [24.3, 25.0]}),
    "sill": pd.DataFrame({"trial": [1], "Q_L_min": [45.0], "y1_mm": [65.0],
                          "y2_mm": [42.0], "delta_measured_mm": [4.0]}),
    "jump": pd.DataFrame({"trial": [1], "H_mm": [75.0], "Q_L_min": [25.0],
                          "z1_mm": [10.0], "z2_mm": [27.0]}),
}

EXAMPLE_WARNING = (
    "⚠️ EXAMPLE DATA: these results use the built-in example values, not your measurements. "
    "Set USE_EXAMPLE_DATA = False and enter your own data before using anything in your report."
)

def use_example_if_requested(df, key, measured_cols):
    """Return the example table when USE_EXAMPLE_DATA is True; otherwise check the table you entered."""
    if USE_EXAMPLE_DATA:
        print(EXAMPLE_WARNING)
        return EXAMPLE_DATA[key].copy()
    if df[measured_cols].isna().all().all():
        print("No measurements entered yet: replace the np.nan entries with your readings.")
    else:
        ex = EXAMPLE_DATA[key]
        if df.shape == ex.shape and np.allclose(df[measured_cols].to_numpy(float),
                                                ex[measured_cols].to_numpy(float), equal_nan=True):
            print("⚠️ These values are identical to the example data. Enter your own measurements.")
    return df

## Section A — Bernoulli verification

You will compare the velocity obtained from the measured flow rate with the ideal velocity predicted from Bernoulli-type reasoning based on the recorded water levels.

For the measured velocity,

$$
V =
rac{Q}{bh}
$$

where $b$ is the flume width and $h$ is the local water depth.

Use the table below for your Bernoulli verification data.

In [ ]:
# Enter your measurements: replace each np.nan with your reading (add rows as needed)
bernoulli_df = pd.DataFrame({
    "trial": [1, 2],
    "H_mm": [np.nan, np.nan],
    "h_mm": [np.nan, np.nan],
    "Q_L_min": [np.nan, np.nan],
})
bernoulli_df = use_example_if_requested(bernoulli_df, "bernoulli", ["H_mm", "h_mm", "Q_L_min"])
bernoulli_df

In [ ]:
bernoulli_df["H_m"] = mm_to_m(bernoulli_df["H_mm"])
bernoulli_df["h_m"] = mm_to_m(bernoulli_df["h_mm"])
bernoulli_df["Q_m3_s"] = bernoulli_df["Q_L_min"].apply(L_min_to_m3_s)

bernoulli_df["V_measured_m_s"] = velocity_rectangular_channel(
    bernoulli_df["Q_m3_s"], flume_width, bernoulli_df["h_m"]
)

# Simple ideal estimate often used for comparison in this lab context
bernoulli_df["V_theory_m_s"] = np.sqrt((2 * g * bernoulli_df["H_m"]**2)/( (bernoulli_df["H_m"] + bernoulli_df["h_m"])))
bernoulli_df["percent_error"] = safe_percent_error(
    bernoulli_df["V_measured_m_s"], bernoulli_df["V_theory_m_s"]
)

bernoulli_df

In [ ]:
valid = bernoulli_df.dropna(subset=["V_measured_m_s", "V_theory_m_s"])

if len(valid) > 0:
    plt.figure()
    plt.plot(valid["trial"], valid["V_measured_m_s"], marker="o", linewidth=2, label="Measured")
    plt.plot(valid["trial"], valid["V_theory_m_s"], marker="s", linewidth=2, label="Theoretical")
    plt.xlabel("Trial")
    plt.ylabel("Velocity (m/s)")
    plt.title("Bernoulli Verification")
    plt.legend()
    save_fig("A_bernoulli_verification")
    plt.show()
else:
    print("Enter at least one complete trial first.")


### Discussion prompts
- Is the measured velocity close to the ideal prediction?
- If not, do losses, nonuniform flow, or reading uncertainty explain the difference?
- Was the flow fully stabilized before each measurement?


## Section B — Submerged sill analysis

## Required formulae

From continuity in a rectangular channel,

$$
Q = b y_1 V_1 = b y_2 V_2
$$

so

$$
V_2 = V_1 \frac{y_1}{y_2}
$$

Applying Bernoulli between the two free-surface locations shown in the lab figure gives the sill dip relation:

$$
\Delta = \frac{V_1^2}{2g}\left[\left(\frac{y_1}{y_2}\right)^2 - 1\right]
$$

and the upstream velocity from the measured flow rate is

$$
V_1 = \frac{Q}{b y_1}
$$

This section computes:
- the measured upstream velocity from flow rate,
- the downstream velocity from continuity,
- the theoretical dip using the exact sill-dip formula,
- the percent difference between measured and theoretical dip.

In [ ]:
# Enter your measurements: replace each np.nan with your reading (add rows as needed)
sill_df = pd.DataFrame({
    "trial": [1],
    "Q_L_min": [np.nan],
    "y1_mm": [np.nan],
    "y2_mm": [np.nan],
    "delta_measured_mm": [np.nan],
})
sill_df = use_example_if_requested(sill_df, "sill", ["Q_L_min", "y1_mm", "y2_mm", "delta_measured_mm"])
sill_df

In [ ]:

sill_df["Q_m3_s"] = sill_df["Q_L_min"].apply(L_min_to_m3_s)
sill_df["y1_m"] = mm_to_m(sill_df["y1_mm"])
sill_df["y2_m"] = mm_to_m(sill_df["y2_mm"])
sill_df["delta_measured_m"] = mm_to_m(sill_df["delta_measured_mm"])

sill_df["V1_m_s"] = velocity_rectangular_channel(
    sill_df["Q_m3_s"], flume_width, sill_df["y1_m"]
)
sill_df["V2_m_s"] = velocity_rectangular_channel(
    sill_df["Q_m3_s"], flume_width, sill_df["y2_m"]
)

sill_df["delta_theory_m"] = (
    sill_df["V1_m_s"]**2 / (2 * g)
) * ((sill_df["y1_m"] / sill_df["y2_m"])**2 - 1)

sill_df["delta_theory_mm"] = 1000 * sill_df["delta_theory_m"]
sill_df["delta_error_percent"] = safe_percent_error(
    sill_df["delta_measured_m"], sill_df["delta_theory_m"]
)

sill_df


In [ ]:
valid = sill_df.dropna(subset=["delta_measured_m", "delta_theory_m"])

if len(valid) > 0:
    plt.figure()
    x = np.arange(len(valid))
    width = 0.35
    plt.bar(x - width/2, valid["delta_measured_mm"], width=width, label="Measured dip")
    plt.bar(x + width/2, valid["delta_theory_mm"], width=width, label="Theoretical dip")
    plt.xticks(x, [f"Trial {int(t)}" for t in valid["trial"]])
    plt.ylabel("Dip, Δ (mm)")
    plt.title("Submerged Sill: Measured vs Theoretical Dip")
    plt.legend()
    save_fig("B_submerged_sill")
    plt.show()
else:
    print("Enter at least one complete sill trial first.")


### Short derivation outline for your write-up
1. Apply Bernoulli between the two free-surface points.
2. Replace velocities using continuity: $b y_1 V_1 = b y_2 V_2$.
3. Rearrange for the free-surface drop $\Delta$.
4. Substitute the measured flow-rate relation $V_1 = Q/(b y_1)$ if needed.


## Section C — Hydraulic jump

For rectangular open-channel flow, the upstream Froude number is

$$
Fr_1 = \frac{V_1}{\sqrt{g z_1}}
$$

and the ideal conjugate-depth relation is

$$
\frac{z_2}{z_1} = \frac{1}{2}\left(-1 + \sqrt{1 + 8 Fr_1^2}\right)
$$

The notebook also compares specific energy before and after the jump.

### Key Equations
* **Froude Number ($Fr$):** $Fr = \frac{v}{\sqrt{gz}}$.
* **Conjugate Depth Ratio:** $\frac{z_2}{z_1} = \frac{1}{2}(-1 + \sqrt{1 + 8Fr_1^2})$.
* **Head Loss ($h_L$):** $\frac{h_L}{z_1} = 1 - \frac{z_2}{z_1} + \frac{Fr_1^2}{2} [1 - (\frac{z_1}{z_2})^2]$.

In [ ]:
# Enter your measurements: replace each np.nan with your reading (add rows as needed)
jump_df = pd.DataFrame({
    "trial": [1],
    "H_mm": [np.nan],
    "Q_L_min": [np.nan],
    "z1_mm": [np.nan],
    "z2_mm": [np.nan],
})
jump_df = use_example_if_requested(jump_df, "jump", ["H_mm", "Q_L_min", "z1_mm", "z2_mm"])
jump_df

In [ ]:
jump_df["H_m"] = mm_to_m(jump_df["H_mm"])
jump_df["Q_m3_s"] = jump_df["Q_L_min"].apply(L_min_to_m3_s)
jump_df["z1_m"] = mm_to_m(jump_df["z1_mm"])
jump_df["z2_m"] = mm_to_m(jump_df["z2_mm"])

jump_df["V1_m_s"] = velocity_rectangular_channel(jump_df["Q_m3_s"], flume_width, jump_df["z1_m"])
jump_df["V2_m_s"] = velocity_rectangular_channel(jump_df["Q_m3_s"], flume_width, jump_df["z2_m"])

# Fr1 from measured velocity and depth
jump_df["Fr1"] = froude_number(jump_df["V1_m_s"], jump_df["z1_m"])

# Fr1 back-calculated from conjugate depths alone
jump_df["Fr1_from_depths"] = np.sqrt(
    (1/8) * ((2 * jump_df["z2_m"] / jump_df["z1_m"] + 1)**2 - 1)
)

# Percent difference between the two Fr1 estimates
jump_df["Fr1_error_percent"] = safe_percent_error(
    jump_df["Fr1_from_depths"], jump_df["Fr1"]
)

jump_df["z2_over_z1_measured"] = jump_df["z2_m"] / jump_df["z1_m"]
jump_df["z2_over_z1_theory"] = 0.5 * (-1 + np.sqrt(1 + 8 * jump_df["Fr1"]**2))

jump_df["E1_m"] = jump_df["z1_m"] + jump_df["V1_m_s"]**2 / (2 * g)
jump_df["E2_m"] = jump_df["z2_m"] + jump_df["V2_m_s"]**2 / (2 * g)
jump_df["head_loss_m"] = jump_df["E1_m"] - jump_df["E2_m"]

is_critical = np.isclose(jump_df["Fr1"], 1.0, rtol=0.0, atol=1e-6)
jump_df["jump_type"] = np.select(
    [
        jump_df["Fr1"] < 1,
        is_critical,
        (jump_df["Fr1"] > 1) & (jump_df["Fr1"] < 1.7),
        (jump_df["Fr1"] >= 1.7) & (jump_df["Fr1"] < 2.5),
        (jump_df["Fr1"] >= 2.5) & (jump_df["Fr1"] < 4.5),
        (jump_df["Fr1"] >= 4.5) & (jump_df["Fr1"] < 9.0),
        jump_df["Fr1"] >= 9.0,
    ],
    [
        "No jump / subcritical upstream",
        "Critical flow (limiting condition; no developed jump)",
        "Standing wave or undular jump",
        "Weak jump",
        "Oscillating jump",
        "Stable steady jump",
        "Strong jump",
    ],
    default="Unclassified"
)

jump_df

In [ ]:
valid = jump_df.dropna(subset=["E1_m", "E2_m"])

if len(valid) > 0:
    first = valid.iloc[0]
    plt.figure()
    plt.bar(["Before jump", "After jump"], [first["E1_m"], first["E2_m"]])
    plt.ylabel("Specific energy (m)")
    plt.title("Hydraulic Jump Energy Comparison")
    save_fig("C_hydraulic_jump_energy")
    plt.show()
else:
    print("Enter at least one complete jump trial first.")

## Section D — Report-ready summary

Run the next cell after you fill in the measurement tables. It prints a concise summary that you can adapt directly into your discussion or conclusion section.

In [ ]:
if USE_EXAMPLE_DATA:
    display(Markdown(f"**{EXAMPLE_WARNING}**"))

summary_lines = []

b = bernoulli_df.dropna(subset=["V_measured_m_s", "V_theory_m_s"])
if len(b) > 0:
    avg_abs_error = np.abs(b["percent_error"]).mean()
    summary_lines.append("## Bernoulli verification")
    summary_lines.append(
        f"The average absolute percent difference between measured and theoretical velocity was {avg_abs_error:.2f}%."
    )
    if avg_abs_error < 10:
        summary_lines.append(
            "The agreement is reasonably good, which suggests Bernoulli's equation provides a useful first-order model for the measured flow."
        )
    else:
        summary_lines.append(
            "The discrepancy suggests that non-ideal effects such as losses, nonuniform velocity distribution, and measurement uncertainty were significant."
        )

s = sill_df.dropna(subset=["delta_measured_m", "delta_theory_m"])
if len(s) > 0:
    avg_abs_error = np.abs(s["delta_error_percent"]).mean()
    summary_lines.append("## Submerged sill")
    summary_lines.append(
        f"Using the exact sill-dip relation, the average absolute percent difference between measured and theoretical dip was {avg_abs_error:.2f}%."
    )
    summary_lines.append(
        "Any remaining difference may be due to free-surface reading uncertainty, local turbulence near the sill, and the assumptions built into the Bernoulli-based derivation."
    )

j = jump_df.dropna(subset=["Fr1", "head_loss_m"])
if len(j) > 0:
    first = j.iloc[0]
    summary_lines.append("## Hydraulic jump")
    summary_lines.append(
        f"For the first complete trial, the upstream Froude number was {first['Fr1']:.2f} "
        f"(from measured flow rate and depth), corresponding to a '{first['jump_type']}' classification."
    )
    # New: cross-check Fr1 from conjugate depths
    if "Fr1_from_depths" in j.columns and "Fr1_error_percent" in j.columns:
        summary_lines.append(
            f"Back-calculating Fr\u2081 from the measured conjugate depths alone gives "
            f"Fr\u2081 = {first['Fr1_from_depths']:.2f}, a difference of "
            f"{abs(first['Fr1_error_percent']):.1f}% from the flow-rate estimate. "
            "This discrepancy reflects non-ideal behavior in the jump: energy losses, "
            "a non-uniform velocity profile, and free-surface reading uncertainty all "
            "cause the real depth ratio to differ from the ideal conjugate-depth prediction."
        )
    summary_lines.append(
        f"The estimated head loss across the jump was {first['head_loss_m']:.4f} m of water."
    )
    # New: energy dissipation as a percentage of upstream specific energy
    if "E1_m" in j.columns:
        energy_loss_pct = 100 * first["head_loss_m"] / first["E1_m"]
        summary_lines.append(
            f"This corresponds to {energy_loss_pct:.1f}% of the upstream specific energy, "
            "consistent with the expectation that hydraulic jumps are an effective but lossy "
            "mechanism for transitioning from supercritical to subcritical flow."
        )
    else:
        summary_lines.append(
            "This is consistent with the expectation that hydraulic jumps dissipate mechanical "
            "energy while increasing depth in the flow direction."
        )

if summary_lines:
    display(Markdown("\n".join(summary_lines)))
else:
    print("Fill in at least one complete data row before generating the summary.")

## Section E — Save figures and tables for your report
Run this cell after the analysis. It saves every figure above as a 300-dpi PNG and every results table as a CSV file, then downloads them together as `lab1_outputs.zip`. If your browser blocks the download, open the **Files** panel on the left and download the zip from there.

In [ ]:
# Results tables to save -- file name: (results table, the measurement table it comes from)
TABLES = {
    "A_bernoulli": ("bernoulli_df", "bernoulli_df"),
    "B_submerged_sill": ("sill_df", "sill_df"),
    "C_hydraulic_jump": ("jump_df", "jump_df")
}

saved_tables = []
def has_measurements(df):
    """True if any measurement column (all columns after the first) holds a number."""
    return isinstance(df, pd.DataFrame) and df.iloc[:, 1:].select_dtypes("number").notna().any().any()

for name, (var, source) in TABLES.items():
    table = globals().get(var)
    if isinstance(table, pd.DataFrame) and has_measurements(globals().get(source)):
        table.to_csv(OUT_DIR / f"{name}.csv", index=False)
        saved_tables.append(f"{name}.csv")

saved_figures = sorted(p.name for p in OUT_DIR.glob("*.png"))
if globals().get("USE_EXAMPLE_DATA", False):
    print(EXAMPLE_WARNING)
print("Figures:", ", ".join(saved_figures) if saved_figures else "none yet (run the plotting cells first)")
print("Tables: ", ", ".join(saved_tables) if saved_tables else "none yet (enter your measurements first)")

zip_path = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print(f"Not running in Colab; outputs saved to {zip_path}")

## Section F — Submission checklist

Before submitting, consider adding:
- your raw data table from the lab,
- figure captions under each plot,
- one short uncertainty paragraph,
- one short conclusion paragraph,
- any required TA-specific derivation steps.

This notebook is meant to save you time on the calculations so you can focus on the interpretation.